# 🐶 Dog vs Cat Image Classification 🐱

Transfer learning with **MobileNetV2** (ImageNet weights) on a subset of the
**Oxford-IIIT Pet Dataset**, mapped to binary labels (Cat / Dog) via breed names.

This notebook mirrors `train_model.py` in compact form — the script is the
full reproducible pipeline (automatic dataset download + subset preparation);
the first code cell below prepares the same subset if it does not exist yet.

**Measured result of the full pipeline run:** see `eval_report.json`
(test accuracy and confusion counts from the held-out test split).

In [ ]:
import os, random, tarfile, urllib.request

CAT_BREEDS = {"Abyssinian", "Bengal", "Birman", "Bombay", "British_Shorthair",
              "Egyptian_Mau", "Maine_Coon", "Persian", "Ragdoll",
              "Russian_Blue", "Siamese", "Sphynx"}
DATA_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"

def breed_of(fn):
    return os.path.splitext(os.path.basename(fn))[0].rsplit("_", 1)[0]

# Download + build data/pets_subset/{cat,dog} with 1,500 images per class
# (seed-fixed subsample). Skipped automatically if the subset exists.
if not os.path.isdir("data/pets_subset/cat"):
    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/images.tar.gz"):
        urllib.request.urlretrieve(DATA_URL, "data/images.tar.gz")
    rng = random.Random(42)
    by_class = {"cat": [], "dog": []}
    with tarfile.open("data/images.tar.gz", "r:gz") as tar:
        members = {m.name: m for m in tar.getmembers()
                   if m.name.startswith("images/") and m.name.endswith(".jpg")}
        for name in members:
            by_class["cat" if breed_of(name) in CAT_BREEDS else "dog"].append(name)
        for cls in ("cat", "dog"):
            rng.shuffle(by_class[cls])
            out = f"data/pets_subset/{cls}"
            os.makedirs(out, exist_ok=True)
            for name in by_class[cls][:1500]:
                with open(os.path.join(out, os.path.basename(name)), "wb") as o:
                    o.write(tar.extractfile(members[name]).read())
print("subset ready")

In [ ]:
import tensorflow as tf

IMG_SIZE, BATCH = 128, 32
tf.keras.utils.set_random_seed(42)

raw = tf.keras.utils.image_dataset_from_directory(
    "data/pets_subset", label_mode="binary", image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, shuffle=True, seed=42)
n = sum(1 for _ in raw.unbatch())
n_test, n_val = int(0.15 * n), int(0.15 * n)
test_ds = raw.take(n_test // BATCH + 1)
rest = raw.skip(n_test // BATCH + 1)
val_ds = rest.take(n_val // BATCH + 1)
train_ds = rest.skip(n_val // BATCH + 1)
print(f"total={n} train={n - n_val - n_test} val={n_val} test={n_test}")

In [ ]:
# MobileNetV2 transfer learning: frozen base + dense head.
# Preprocessing lives INSIDE the model (a Rescaling layer = MobileNetV2's
# preprocess_input, x / 127.5 - 1.0) so the saved model takes plain
# [0, 255] pixels — identical preprocessing at inference time.
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

aug = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"),
                           tf.keras.layers.RandomRotation(0.1),
                           tf.keras.layers.RandomZoom(0.1)])
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = aug(inputs)
x = tf.keras.layers.Rescaling(scale=1.0 / 127.5, offset=-1.0,
                              name="mobilenet_preprocess")(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# Phase 1: train the head (6 epochs). Phase 2: fine-tune top 30 base
# layers at a low learning rate (3 epochs).
history = model.fit(train_ds, validation_data=val_ds, epochs=6)

base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="binary_crossentropy", metrics=["accuracy"])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=9, initial_epoch=6)

In [ ]:
import numpy as np

# Evaluate on the held-out test split (class order: cat=0, dog=1)
y_true, y_prob = [], []
for images, labels in test_ds:
    y_prob.extend(model.predict(images, verbose=0).flatten().tolist())
    y_true.extend(labels.numpy().flatten().tolist())
y_true = np.array(y_true, dtype=int)
y_pred = (np.array(y_prob) >= 0.5).astype(int)
acc = (y_true == y_pred).mean()
print(f"Test accuracy: {acc:.4f} on {len(y_true)} images")
print("cat correct:", int(((y_true == 0) & (y_pred == 0)).sum()),
      "| cat as dog:", int(((y_true == 0) & (y_pred == 1)).sum()),
      "| dog correct:", int(((y_true == 1) & (y_pred == 1)).sum()),
      "| dog as cat:", int(((y_true == 1) & (y_pred == 0)).sum()))

model.save("model.keras")
print("saved model.keras")